# 01 — NHANES Cardiovascular Data Preparation

**Project:** Cardiovascular Risk & Clinical Insights  
**Data:** CDC NHANES 2017–2018, six public-use XPT files.  
**Output:** `data/processed/cardiovascular_data.csv` and `data/processed/data_quality_summary.csv`.

This notebook **only** downloads, joins, defines variables and audits the analytical dataset. Visualisations and clinical interpretation belong to notebook 02.

Run all cells top to bottom using your project's `.venv` kernel. The initial download requires network access; files already saved in `data/raw` are reused. No raw data are bundled.

**Methodological caution:** NHANES is a cross-sectional, stratified multistage probability survey. These preparations do not establish treatment effects or diagnoses. The BP definition below is a descriptive operational threshold, not a clinical diagnosis.

## 1. Project paths and imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests

BASE = Path.cwd().resolve()
if BASE.name.lower() == 'notebooks':
    BASE = BASE.parent
if not (BASE / 'requirements.txt').exists():
    raise FileNotFoundError('Open this notebook from cardiovascular-nhanes/notebooks/ or the project root.')
RAW = BASE / 'data' / 'raw'
OUT = BASE / 'data' / 'processed'
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
print('Project root:', BASE)
print('Data cache:', RAW)

## 2. Source data: download/cache and read

If CDC access is restricted, download the six `.XPT` files manually from [NHANES 2017–2018](https://wwwn.cdc.gov/nchs/nhanes/search/DataPage.aspx?Cycle=2017-2018) and place them in `data/raw/`.

In [ ]:
FILES = ['DEMO_J', 'BPX_J', 'BMX_J', 'TCHOL_J', 'HDL_J', 'BPQ_J']

def get_xpt(stem):
    dest = RAW / f'{stem}.XPT'
    if dest.is_file() and dest.stat().st_size > 1000:
        print('Cached:', stem)
        return dest
    urls = [
        f'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/{stem}.xpt',
        f'https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/{stem}.XPT',
    ]
    failures = []
    for url in urls:
        try:
            response = requests.get(url, timeout=35)
            response.raise_for_status()
            if len(response.content) < 1000 or b'<html' in response.content[:300].lower():
                raise ValueError('Invalid or HTML response')
            dest.write_bytes(response.content)
            print('Downloaded:', stem)
            return dest
        except (requests.RequestException, ValueError) as error:
            failures.append(str(error)[:100])
    raise RuntimeError(f'Could not retrieve {stem}. Please save its XPT in {RAW}. Details: {failures}')

files = {name: get_xpt(name) for name in FILES}
raw = {name: pd.read_sas(path, format='xport') for name, path in files.items()}
for name, df in raw.items():
    assert 'SEQN' in df and df['SEQN'].is_unique, f'Invalid IDs in {name}'
    print(f'{name}: {len(df):,} participants, {df.shape[1]} columns')

## 3. Merge participant-level datasets

`SEQN` is the public NHANES participant identifier. Start with demographic respondents and use left one-to-one merges to preserve the cohort. A left merge does not imply that every participant has every measurement.

In [ ]:
SELECT = {
    'DEMO_J': ['SEQN','RIDAGEYR','RIAGENDR','WTMEC2YR','SDMVPSU','SDMVSTRA'],
    'BPX_J': ['SEQN'] + [f'BPXSY{i}' for i in range(1,5)] + [f'BPXDI{i}' for i in range(1,5)],
    'BMX_J': ['SEQN','BMXBMI'],
    'TCHOL_J': ['SEQN','LBXTC'],
    'HDL_J': ['SEQN','LBDHDD'],
    'BPQ_J': ['SEQN','BPQ020','BPQ050A'],
}
for name, fields in SELECT.items():
    missing = set(fields) - set(raw[name].columns)
    if missing:
        raise KeyError(f'{name} lacks {sorted(missing)}')
combined = raw['DEMO_J'][SELECT['DEMO_J']].copy()
for name in FILES[1:]:
    combined = combined.merge(raw[name][SELECT[name]], on='SEQN', how='left', validate='one_to_one')
assert combined.SEQN.is_unique
print('Participants (all ages):', len(combined))

## 4. Define the adult cohort and derived variables

- Adult cohort: age ≥20 years.
- Use the arithmetic mean of **available plausible BP readings**, including reading 4 where recorded; require at least one valid measurement for each pressure type.
- Plausibility filter for this portfolio project: systolic 60–300 mmHg; diastolic 30–200 mmHg. **These are project assumptions, not NHANES official cleaning rules**; we retain excluded-reading counts. In particular, zero diastolic readings warrant sensitivity analysis in future work.
- `high_measured_bp` is the indicator (SBP ≥130 or DBP ≥80) only when **both** calculated BP components exist; this is not a diagnosis.
- BP questionnaire codes 1=yes, 2=no; other/missing responses remain missing. `BPQ050A` is subject to questionnaire skip logic: missing **does not** mean 'No'.
- We retain NHANES two-year MEC weights, strata and PSUs for later survey-aware analysis.

In [ ]:
adults = combined.loc[combined['RIDAGEYR'] >= 20].copy()

for prefix, lower, upper, out in [('BPXSY',60,300,'sbp'), ('BPXDI',30,200,'dbp')]:
    cols = [f'{prefix}{i}' for i in range(1,5)]
    readings = adults[cols].apply(pd.to_numeric, errors='coerce')
    valid = readings.where((readings >= lower) & (readings <= upper))
    adults[f'{out}_mean'] = valid.mean(axis=1)
    adults[f'{out}_n_readings'] = valid.notna().sum(axis=1)
    adults[f'{out}_excluded_readings'] = (readings.notna() & valid.isna()).sum(axis=1)

has_both = adults['sbp_mean'].notna() & adults['dbp_mean'].notna()
adults['high_measured_bp'] = np.where(
    has_both,
    ((adults['sbp_mean'] >= 130) | (adults['dbp_mean'] >= 80)).astype(int),
    np.nan,
)
adults['sex'] = adults['RIAGENDR'].map({1: 'Male', 2: 'Female'})
adults['age_group'] = pd.cut(adults['RIDAGEYR'], [19,39,59,79,np.inf], labels=['20–39','40–59','60–79','80+'])
adults['bmi'] = adults['BMXBMI']
adults['total_cholesterol_mg_dl'] = adults['LBXTC']
adults['hdl_mg_dl'] = adults['LBDHDD']
adults['told_hypertension'] = adults['BPQ020'].map({1:'Yes',2:'No'})
adults['bp_medication'] = adults['BPQ050A'].map({1:'Yes',2:'No'})
adults['mec_weight'] = adults['WTMEC2YR']
print('Adults:',len(adults))
print('SBP available:',adults.sbp_mean.notna().sum())
print('DBP available:',adults.dbp_mean.notna().sum())
print('Both BP types available:',has_both.sum())
print('Medication status missing / not applicable:',adults.bp_medication.isna().sum())

## 5. Cohort and missingness audit

Missing questionnaire information is not imputed; analyses will use variable-specific subsets. Document the number of observations available for each intended comparison.

In [ ]:
audit_fields = ['RIDAGEYR','sex','sbp_mean','dbp_mean','bmi',
                'total_cholesterol_mg_dl','hdl_mg_dl','bp_medication',
                'mec_weight','SDMVSTRA','SDMVPSU']
audit = pd.DataFrame({
    'available_n': adults[audit_fields].notna().sum(),
    'missing_n': adults[audit_fields].isna().sum(),
    'missing_pct': (100 * adults[audit_fields].isna().mean()).round(1),
})
display(audit)
print('Unique adult records:', adults.SEQN.nunique())
assert adults.SEQN.is_unique
assert (adults['sbp_mean'].dropna().between(60,300)).all()
assert (adults['dbp_mean'].dropna().between(30,200)).all()

## 6. Save outputs for the next notebook

Both datasets are **local-only** according to `.gitignore`. Run notebook 02 in a separate session: it reads the exported CSV rather than relying on the memory state of this notebook.

In [ ]:
EXPORT = ['SEQN','RIDAGEYR','sex','age_group','sbp_mean','dbp_mean',
          'sbp_n_readings','dbp_n_readings','sbp_excluded_readings','dbp_excluded_readings',
          'high_measured_bp','bmi','total_cholesterol_mg_dl','hdl_mg_dl',
          'told_hypertension','bp_medication','mec_weight','SDMVSTRA','SDMVPSU']
output_file = OUT / 'cardiovascular_data.csv'
adults[EXPORT].to_csv(output_file, index=False)
audit.to_csv(OUT / 'data_quality_summary.csv')
print('Saved:', output_file)
print('Rows:',len(adults),'Columns:',len(EXPORT))
assert output_file.is_file()

## Next step

Open `02_exploratory_analysis.ipynb` and execute it from top to bottom. It reads `data/processed/cardiovascular_data.csv`.

**Source references:** [NHANES 2017–2018 datasets](https://wwwn.cdc.gov/nchs/nhanes/search/DataPage.aspx?Cycle=2017-2018), [blood pressure examination codebook](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BPX_J.htm), [blood pressure questionnaire codebook](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BPQ_J.htm).